# T14 - 债券框架

## 项目概述

债券框架是一个综合性的债券分析工具集，主要用于债券收益率定价检验和曲线选项设计。该项目提供了完整的债券市场数据分析框架，包括数据获取、标准化处理、统计分析和可视化展示。

### 主要功能
- 债券收益率定价检验
- 曲线选项设计
- 海外债基研究
- 因子解释度分析
- 政策预期分析

### 应用场景
- 债券收益率预测
- 因子有效性检验
- 投资组合优化
- 风险评估

---

## 1. 环境配置

In [ ]:
# 1.1 导入标准库
import os
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# 添加项目路径
PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"项目目录: {PROJECT_DIR}")

In [ ]:
# 1.2 导入数据处理库
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm

print(f"pandas版本: {pd.__version__}")
print(f"numpy版本: {np.__version__}")

In [ ]:
# 1.3 导入统计分析库
import statsmodels.api as sm
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from numpy.linalg import LinAlgError

print("统计分析库导入成功")

In [ ]:
# 1.4 导入可视化库
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("可视化库导入成功")

In [ ]:
# 1.5 导入数据库连接库
import sqlalchemy
from sqlalchemy.pool import NullPool

print("数据库连接库导入成功")

In [ ]:
# 1.6 导入项目配置
import config
from src.utils import get_sql_engine, get_yield_curve_data, get_standardized_data
from src.utils import load_financial_data, load_yield_data, get_curve_options_data, build_curve_options_json
from src.analysis import batch_regression, build_analysis_dataset, run_regression_analysis
from src.analysis import rolling_regression, run_regression_advanced, calculate_optimal_shift
from src.visualization import create_dynamic_html, plot_comparison, plot_dynamic_comparison
from src.visualization import visualize_results, plot_pgim_fund_data

print("项目模块导入成功")

In [ ]:
# 1.7 配置环境变量（示例）
# 实际使用时请设置环境变量，不要在代码中硬编码密码
# os.environ['DB_PASSWORD'] = 'your_password_here'

# 检查环境变量
required_env_vars = ['DB_PASSWORD']
missing_vars = [v for v in required_env_vars if not os.environ.get(v)]

if missing_vars:
    print(f"警告: 缺少环境变量 {missing_vars}，请设置后再运行数据库相关操作")
else:
    print("环境变量配置完整")

---

## 2. 数据获取

In [ ]:
# 2.1 查看收益率曲线代码配置
print("收益率曲线代码配置:")
for code, name in config.YIELD_CURVE_CODES.items():
    print(f"  {code}: {name}")

In [ ]:
# 2.2 获取国债收益率数据（示例）
# 如果数据库密码已配置，取消下面的注释运行

# try:
#     yield_data = get_yield_curve_data('L001619604', start_date='2020-01-01')
#     print(f"获取到 {len(yield_data)} 条10年期国债收益率数据")
#     print(yield_data.head())
# except Exception as e:
#     print(f"获取数据失败: {e}")

print("请配置数据库密码后运行此单元格")

In [ ]:
# 2.3 获取标准化因子数据（示例）
# 标准化取数函数会剔除异常值（3倍标准差）

# try:
#     factor_data = get_standardized_data('M0000612', start_date='2020-01-01')
#     print(f"获取到 {len(factor_data)} 条CPI数据")
#     print(factor_data.head())
# except Exception as e:
#     print(f"获取数据失败: {e}")

print("请配置数据库密码后运行此单元格")

In [ ]:
# 2.4 获取曲线选项数据
# 用于生成多级联动的曲线选择器

# try:
#     df_curve = get_curve_options_data()
#     curve_options = build_curve_options_json(df_curve)
#     print(f"获取到 {len(df_curve)} 条曲线信息")
#     print(f"曲线类型数量: {len(curve_options['_series'])}")
# except Exception as e:
#     print(f"获取数据失败: {e}")

print("请配置数据库密码后运行此单元格")

In [ ]:
# 2.5 加载资金因子数据

# try:
#     fund_data = load_financial_data('L001619493', start_date='2020-01-01')
#     print(f"DR007数据时间范围: {fund_data.index.min()} ~ {fund_data.index.max()}")
#     print(fund_data.head())
# except Exception as e:
#     print(f"获取数据失败: {e}")

print("请配置数据库密码后运行此单元格")

---

## 3. 数据处理

In [ ]:
# 3.1 因子数据预处理示例
from src.analysis import preprocess_factor_data

# 创建示例数据
sample_data = pd.DataFrame({
    'date': pd.date_range('2020-01-01', periods=100, freq='D'),
    'value': np.random.randn(100).cumsum() + 100
})

# 预处理同比数据
processed_yoy = preprocess_factor_data(sample_data.copy(), 'CPI:当月同比')
print("同比数据处理后:")
print(processed_yoy.head())

In [ ]:
# 3.2 构建分析数据集（资金因子定价分析）

# try:
#     analysis_df = build_analysis_dataset(
#         factor_code='L001619493',  # DR007
#         yield_code='L001619604'     # 10年国债
#     )
#     print(f"分析数据集行数: {len(analysis_df)}")
#     print(analysis_df.head())
# except Exception as e:
#     print(f"构建数据集失败: {e}")

print("请配置数据库密码后运行此单元格")

In [ ]:
# 3.3 计算最优平移值
# 寻找资金利率与债券收益率之间的最优平移值

# 示例：使用模拟数据演示
np.random.seed(42)
fund_rate = pd.Series(np.random.randn(100).cumsum(), index=pd.date_range('2020-01-01', periods=100))
yield_rate = pd.Series(fund_rate.values * 0.5 + np.random.randn(100) * 0.1, index=fund_rate.index)

optimal_shift, correlation = calculate_optimal_shift(fund_rate, yield_rate)
print(f"最优平移值: {optimal_shift:.4f}")
print(f"相关系数: {correlation:.4f}")

---

## 4. 核心逻辑

In [ ]:
# 4.1 单因子回归分析

# try:
#     # 使用部分因子进行演示
#     demo_factors = dict(list(config.MACRO_ECONOMIC_PRICE_FACTOR_CODES.items())[:5])
#     results = batch_regression('L001619604', demo_factors)
#     print("单因子回归结果:")
#     print(results.head())
# except Exception as e:
#     print(f"回归分析失败: {e}")

print("请配置数据库密码后运行此单元格")

In [ ]:
# 4.2 整体指标分析（PCA降维 + 岭回归）

# try:
#     # 使用部分因子进行演示
#     demo_factors = dict(list(config.MACRO_ECONOMIC_PRICE_FACTOR_CODES.items())[:10])
#     group_results = batch_regression(
#         'L001619604', 
#         demo_factors, 
#         group_type='price',
#         rolling_window=36
#     )
#     print("整体指标分析结果:")
#     print(group_results.head())
# except Exception as e:
#     print(f"分析失败: {e}")

print("请配置数据库密码后运行此单元格")

In [ ]:
# 4.3 资金因子定价回归分析

# try:
#     analysis_df = build_analysis_dataset(
#         factor_code='L001619493',
#         yield_code='L001619604'
#     )
#     reg_results = run_regression_analysis(analysis_df)
#     
#     for name, result in reg_results.items():
#         print(f"\n{name}回归结果:")
#         print(f"R方: {result.rsquared:.4f}")
#         print(f"系数: {result.params.to_dict()}")
# except Exception as e:
#     print(f"回归分析失败: {e}")

print("请配置数据库密码后运行此单元格")

In [ ]:
# 4.4 增强回归模型（处理自相关和异方差）

# try:
#     advanced_results = run_regression_advanced(analysis_df)
#     
#     print("=== 模型对比 ===")
#     for model_name, result in advanced_results.items():
#         dw = sm.stats.durbin_watson(result.resid)
#         print(f"{model_name} DW统计量: {dw:.4f}")
# except Exception as e:
#     print(f"增强回归失败: {e}")

print("请配置数据库密码后运行此单元格")

In [ ]:
# 4.5 滚动回归分析

# try:
#     roll_df = rolling_regression(analysis_df, window=252)
#     print(f"滚动回归结果行数: {len(roll_df)}")
#     print(roll_df.head())
# except Exception as e:
#     print(f"滚动回归失败: {e}")

print("请配置数据库密码后运行此单元格")

---

## 5. 执行与测试

In [ ]:
# 5.1 主分析流程（完整版）
def main_analysis():
    """执行完整的债券收益率定价检验分析"""
    # 定义收益率曲线
    yield_curves = {
        '3年': 'L001618297',
        '10年': 'L001619604',
        '30年': 'L001618299'
    }

    all_results = {}
    dynamic_group_results_all = []
    
    # 单因子分析
    for term, code in yield_curves.items():
        # 价格指标单因子分析
        price_results = batch_regression(code, config.MACRO_ECONOMIC_PRICE_FACTOR_CODES)
        price_results['期限'] = term
        price_results['因子类型'] = '价格指标'
        
        # 数量指标单因子分析
        quantity_results = batch_regression(code, config.MACRO_ECONOMIC_QUANTITY_FACTOR_CODES)
        quantity_results['期限'] = term
        quantity_results['因子类型'] = '数量指标'
        
        # 合并结果
        all_results[term] = pd.concat([price_results, quantity_results])
    
    # 整体指标分析
    group_results = []
    for term, code in yield_curves.items():
        # 动态整体指标分析
        dynamic_price = batch_regression(
            code, config.MACRO_ECONOMIC_PRICE_FACTOR_CODES, 
            group_type='price', rolling_window=36
        )
        dynamic_price['期限'] = term
        dynamic_group_results_all.append(dynamic_price)
        
        dynamic_quantity = batch_regression(
            code, config.MACRO_ECONOMIC_QUANTITY_FACTOR_CODES, 
            group_type='quantity', rolling_window=36
        )
        dynamic_quantity['期限'] = term
        dynamic_group_results_all.append(dynamic_quantity)
    
    # 保存结果
    pd.concat(dynamic_group_results_all).to_csv(
        config.OUTPUT_DIR / '动态整体指标分析结果.csv', index=False
    )
    
    # 可视化
    if all_results:
        plot_comparison(pd.concat(all_results.values()), pd.concat(group_results, ignore_index=True))
    plot_dynamic_comparison(pd.concat(dynamic_group_results_all))
    
    print("分析完成!")
    return all_results, dynamic_group_results_all

# 运行主分析
# all_results, dynamic_results = main_analysis()

print("请配置数据库密码后取消注释运行主分析")

In [ ]:
# 5.2 资金因子定价分析流程
def run_fund_factor_analysis():
    """执行资金因子定价分析"""
    yield_codes = ['L001618297', 'L001619604', 'L001618299']
    
    for yield_code in yield_codes:
        print(f"\n=== 开始分析 yield_code: {yield_code} ===")
        
        # 构建分析数据集
        analysis_df = build_analysis_dataset(
            factor_code='L001619493', 
            yield_code=yield_code
        )
        
        # 回归分析
        reg_results = run_regression_analysis(analysis_df)
        
        # 时变分析
        roll_df = rolling_regression(analysis_df)
        
        # 增强分析
        advanced_results = run_regression_advanced(analysis_df)
        
        # 可视化
        visualize_results(analysis_df, roll_df, advanced_results, yield_code)
        
        # 保存结果
        with pd.ExcelWriter(config.OUTPUT_DIR / f'analysis_data_{yield_code}.xlsx') as writer:
            analysis_df.to_excel(writer, sheet_name='数据')
            pd.DataFrame({
                '模型': ['基础模型', '自回归模型', '误差修正'],
                'DW统计量': [
                    sm.stats.durbin_watson(advanced_results['基础模型'].resid),
                    sm.stats.durbin_watson(advanced_results['自回归模型'].resid),
                    sm.stats.durbin_watson(advanced_results['误差修正模型'].resid)
                ]
            }).to_excel(writer, sheet_name='模型诊断')
        
        print(f"=== 完成分析 yield_code: {yield_code} ===")

# run_fund_factor_analysis()

print("请配置数据库密码后取消注释运行资金因子分析")

In [ ]:
# 5.3 曲线选项生成
def generate_curve_options():
    """生成曲线选项JSON结构"""
    df_curve = get_curve_options_data()
    curve_options = build_curve_options_json(df_curve)
    
    # 保存到文件
    import json
    with open(config.OUTPUT_DIR / 'curve_options.json', 'w', encoding='utf-8') as f:
        json.dump(curve_options, f, ensure_ascii=False, indent=2)
    
    print(f"曲线选项已保存，共 {len(curve_options['_series'])} 个曲线类型")
    return curve_options

# generate_curve_options()

print("请配置数据库密码后取消注释运行曲线选项生成")

In [ ]:
# 5.4 海外债基数据可视化
# PGIM Muni High Income Fund 年度表现

fig = plot_pgim_fund_data()
print("PGIM基金图表已生成到output目录")
fig.show()

---

## 6. 工具函数

In [ ]:
# 6.1 PCA降维演示
from src.analysis import pca_reduce

# 创建示例数据
np.random.seed(42)
demo_data = pd.DataFrame(
    np.random.randn(100, 10),
    columns=[f'factor_{i}' for i in range(10)]
)

reduced_data, var_ratio = pca_reduce(demo_data, n_components=0.95)
print(f"原始特征数: {demo_data.shape[1]}")
print(f"降维后特征数: {reduced_data.shape[1]}")
print(f"累计解释方差比: {var_ratio.sum():.4f}")

In [ ]:
# 6.2 岭回归演示
from src.analysis import ridge_regression

# 创建示例数据
np.random.seed(42)
X = np.random.randn(100, 5)
y = X @ np.array([1, 0.5, -0.3, 0.2, 0.1]) + np.random.randn(100) * 0.1

model = ridge_regression(X, y, alpha=1.0)
print(f"岭回归系数: {model.coef_}")
print(f"R方: {model.score(X, y):.4f}")

In [ ]:
# 6.3 动态HTML报告生成演示
# 使用模拟数据演示

np.random.seed(42)
dates = pd.date_range('2020-01-01', periods=500, freq='D')

analysis_df_demo = pd.DataFrame({
    'dt': dates,
    'fund_rate': np.random.randn(500).cumsum() + 2,
    'yield_rate': np.random.randn(500).cumsum() + 3,
    '预期缺口': np.random.randn(500) * 0.5
})

roll_df_demo = pd.DataFrame({
    'dt': dates[252:],
    'fund_beta': np.random.randn(248) * 0.1 + 0.5,
    'gap_beta': np.random.randn(248) * 0.1 + 0.3
})

create_dynamic_html(
    analysis_df_demo, 
    roll_df_demo, 
    filename=str(config.OUTPUT_DIR / 'demo_analysis_dashboard.html')
)
print(f"演示报告已生成: {config.OUTPUT_DIR / 'demo_analysis_dashboard.html'}")

In [ ]:
# 6.4 查看输出目录
print("输出目录内容:")
for f in config.OUTPUT_DIR.iterdir():
    print(f"  {f.name}")

In [ ]:
# 6.5 清理和总结
print("\n=== 债券框架 Notebook 执行完成 ===")
print(f"项目目录: {PROJECT_DIR}")
print(f"输出目录: {config.OUTPUT_DIR}")
print("\n主要功能:")
print("  1. 债券收益率定价检验")
print("  2. 曲线选项设计")
print("  3. 海外债基研究")
print("  4. 因子解释度分析")
print("  5. 政策预期分析")
print("\n使用前请确保配置数据库密码环境变量 DB_PASSWORD")